# 可编辑的多叉树规则挖掘

`MultiwayTreeExtractor` 直接训练多叉树，支持数值区间、类别分组、逐节点修改、继续生长和剪枝。
本例用真实放款数据，以 FPD 为目标，并按 MOB1 的 7/3/0 天口径输出金额规则报告。
图形复用现有 `plot_tree` 风格。最后演示多分类节点分布。

卡方分裂参考 SAS Enterprise Miner 的 LogWorth；本实现使用 Bonferroni 调整，不声称与 SAS 算法完全相同。
[算法与编辑语义](../docs/multiway_tree.rst)。修改库代码后请重启内核从头运行。

In [ ]:
from pathlib import Path
import sys
import copy

ROOT = Path.cwd() if (Path.cwd() / 'hscredit').exists() else Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.model_selection import train_test_split
from hscredit import MultiwayTreeExtractor, plot_tree
from hscredit.utils import style_rule_table

output_dir = ROOT / 'examples/model_report/multiway_tree'
output_dir.mkdir(parents=True, exist_ok=True)
data = pd.read_excel(ROOT / 'examples/hscredit_yyp.xlsx')
features = ['衡枢鉴真分老客版', '近六个月非银多头机构数', '青云24', '商品类别']
train, test = train_test_split(data, test_size=0.3, random_state=42, stratify=data['FPD'])
print(f'总样本 {len(data)}，训练 {len(train)}，测试 {len(test)}')

## 1. 自动生成多叉树
`max_branches=4` 是上限，实际分支数由候选分裂价值和样本约束决定。

In [ ]:
tree = MultiwayTreeExtractor(
    target='FPD', features=features, max_branches=4, max_depth=2,
    min_samples_leaf=20, criterion='chi2', alpha=0.2,
    feature_map={'衡枢鉴真分老客版': '老客鉴真风险分'},
).fit(train)
display(tree.get_split_candidates(node=0))
display(tree.get_node_table())
fig = plot_tree(tree, save=str(output_dir / '自动多叉树.png'))
baseline = tree.evaluate({'训练集': train, '测试集': test})
display(baseline)

## 2. 人工指定三个阈值，直接生成四个分支
区间为左开右闭，缺失值可指定进入某个分支。操作会替换当前节点后代，其他子树不变。

In [ ]:
edited = copy.deepcopy(tree)
edited.manual_split(feature=features[0], thresholds=[0.12, 0.18, 0.25], node=0, missing_branch=0)
assert len(edited.nodes_[0].children) == 4
fig = plot_tree(edited, save=str(output_dir / '手工四叉树.png'))
display(style_rule_table(edited.get_rule_table(leaf_only=True)))

## 3. 调整阈值、继续分裂、剪枝
节点编号从 `nodes_` 或节点表中读取，不假定编号连续。人工空分支保留 0 样本，预测继承非空祖先概率。

In [ ]:
edited.set_thresholds(0, [0.10, 0.20, 0.30], missing_branch=0)
child = max(edited.nodes_[0].children, key=lambda node: edited.nodes_[node].counts.sum())
edited.grow(node=child)
fig = plot_tree(edited, save=str(output_dir / '继续生长.png'))
display(edited.get_node_table())
edited.prune(node=child)
assert not edited.nodes_[child].children
fig = plot_tree(edited, save=str(output_dir / '节点剪枝.png'))

## 4. 类别分组也可以人工调整
未知类别跟随保存的缺失分支；下面用商品类别演示完整分组。

In [ ]:
category_tree = copy.deepcopy(tree)
categories = data['商品类别'].dropna().unique().tolist()
train_categories = train['商品类别'].dropna().unique().tolist()
if len(train_categories) >= 2:
    groups = [[value] for value in train_categories[:3]]
    if len(train_categories) > 3:
        groups.append(train_categories[3:])
    category_tree.manual_split(feature='商品类别', category_groups=groups, node=0, missing_branch=0)
    display(category_tree.get_node_table())
    fig = plot_tree(category_tree, save=str(output_dir / '类别多叉树.png'))

## 5. 规则集与模型效果
全叶规则互斥且覆盖全部样本。业务拒绝策略应明确挑选风险叶节点，而不是将所有叶子做 OR。

In [ ]:
leaf_table = edited.report(test, leaf_only=True)
display(style_rule_table(leaf_table))
display(pd.concat([baseline.assign(模型='自动多叉树'), edited.evaluate({'训练集': train, '测试集': test}).assign(模型='人工调整树')], ignore_index=True))

rules = edited.get_rules()
masks = np.column_stack([rule.predict(test).to_numpy() for rule in rules])
assert np.all(masks.sum(axis=1) == 1)
assert leaf_table['样本总数'].sum() == len(test)
for rule in rules:
    print(rule.name, rule.expr)

risk_node = int(leaf_table.sort_values('坏样本率', ascending=False).iloc[0]['节点编号'])
strategy = edited.to_ruleset(nodes=[risk_node])
matched, details = strategy.evaluate(test)
print(f'选定风险叶节点 {risk_node}，命中 {matched.sum()} 人')
display(edited.report(test, overdue=['MOB1'], dpds=[7, 3, 0], amount='放款金额', leaf_only=True))

## 6. 保存模型与三种绘图后端
模型包含类别顺序、人工阈值和缺失/未知类别归属。只加载可信来源的模型文件。

In [ ]:
model_path = output_dir / '多叉树.pkl'
edited.save(model_path)
restored = MultiwayTreeExtractor.load(model_path)
np.testing.assert_allclose(restored.predict_proba(test), edited.predict_proba(test))
chart = plot_tree(restored, backend='pyecharts', save=str(output_dir / '多叉树.html'))
dot = plot_tree(restored, backend='graphviz')
(output_dir / '多叉树.gv').write_text(dot.source, encoding='utf-8')
print(f'模型与图形已保存到 {output_dir}')

## 7. 多分类节点分布
多分类节点逐类显示样本数和占比，模型评估提供准确率、交叉熵和 OVR AUC。

In [ ]:
status_data = pd.DataFrame({
    '账龄分组': ['短', '中', '长'] * 40,
    '账户状态': ['正常', '已结清', '逾期'] * 40,
})
status_tree = MultiwayTreeExtractor(target='账户状态', max_branches=4, max_depth=2).fit(status_data)
assert len(status_tree.nodes_[0].children) == 3
fig = plot_tree(status_tree, save=str(output_dir / '多分类三叉树.png'))
display(status_tree.evaluate())
display(status_tree.get_rule_table(leaf_only=True))

## 8. 缺失值独立成一个分支

与 binner 一致，独立分支用 `missing_separate=True` 声明。
三个有限阈值对应四个普通分支，再加缺失分支共五叉，所以设置 `max_branches=5`。
下面只在训练数据副本里注入缺失，原始数据和工作簿保持不变。

In [ ]:
missing_training = train.copy()
missing_training.loc[missing_training.index[:8], features[0]] = np.nan
missing_tree = MultiwayTreeExtractor(
    target='FPD', features=features + ['占信V3'], max_depth=2, max_branches=5,
    missing_separate=True, n_jobs=1,
).fit(missing_training)
missing_tree.manual_split(feature="占信V3", thresholds=[400, 600], node=2, missing_separate=True)
fig = plot_tree(missing_tree, save=str(output_dir / '独立缺失五叉树.png'))
style_rule_table(missing_tree.get_rule_table(leaf_only=True))

In [ ]:
missing_training = train.copy()
missing_training.loc[missing_training.index[:8], features[0]] = np.nan
missing_tree = MultiwayTreeExtractor(
    target='FPD', features=features, max_depth=2, max_branches=5,
    missing_separate=True, n_jobs=1,
).fit(missing_training)
missing_tree.manual_split(feature=features[0], thresholds=[0.12, 0.18, 0.25], node=0, missing_separate=True)
root = missing_tree.nodes_[0]
assert len(root.children) == 5
missing_node = root.children[root.missing_branch]
assert missing_tree.nodes_[missing_node].counts.sum() == missing_training[features[0]].isna().sum()
fig = plot_tree(missing_tree, save=str(output_dir / '独立缺失五叉树.png'))
display(style_rule_table(missing_tree.get_rule_table(leaf_only=True)))

## 9. 将缺失值并入指定分支

数值 `thresholds` 中的 `np.nan` 位置表示合并目标普通分支：位于开头是 0，插在第一个切点之后是 1，依此类推。
显式 `missing_branch` 优先，因此末尾加 `np.nan` 同时传 `missing_branch=0` 也表示合并到第 0 支，不会新增分支。

`[0.12, np.nan, 0.18, 0.25]` 对应 `missing_branch=1`，两种写法比较时必须使用相同分支号。

In [ ]:
# 只需调整 merge_branch，缺失标记、显式参数和原始行验证会同步变化。
cuts = [0.12, 0.18, 0.25]
merge_branch = 1  # 0 为最左分支；1 对应 (0.12, 0.18]
marked_cuts = cuts.copy()
marked_cuts.insert(merge_branch, np.nan)
merged_tree = copy.deepcopy(missing_tree)
merged_tree.manual_split(feature=features[0], thresholds=marked_cuts, node=0)
assert len(merged_tree.nodes_[0].children) == 4
assert not merged_tree.nodes_[0].missing_separate
assert merged_tree.nodes_[0].missing_branch == merge_branch
explicit_tree = copy.deepcopy(missing_tree)
explicit_tree.manual_split(feature=features[0], thresholds=cuts + [np.nan], node=0, missing_branch=merge_branch)
np.testing.assert_allclose(merged_tree.predict_proba(missing_training), explicit_tree.predict_proba(missing_training))
values = missing_training[features[0]]
ordinary_mask = values.notna()
if merge_branch > 0:
    ordinary_mask &= values > cuts[merge_branch - 1]
if merge_branch < len(cuts):
    ordinary_mask &= values <= cuts[merge_branch]
mask = values.isna() | ordinary_mask
merged_node = merged_tree.nodes_[0].children[merge_branch]
assert merged_tree.nodes_[merged_node].counts.sum() == mask.sum()
np.testing.assert_array_equal(merged_tree.apply(missing_training) == merged_node, mask.to_numpy())
fig = plot_tree(merged_tree, save=str(output_dir / '缺失合并四叉树.png'))
display(style_rule_table(merged_tree.get_rule_table(leaf_only=True)))

## 10. 类别分组中的缺失标记

`[np.nan]` 是独立缺失组，`[类别, np.nan]` 将缺失与类别合并。
独立模式下未知类别走样本最多的普通分支，纯缺失分支只接收缺失值。

In [ ]:
category_values = train['商品类别'].dropna().unique().tolist()
ordinary_groups = [[value] for value in category_values[:3]]
if len(category_values) > 3:
    ordinary_groups.append(category_values[3:])
category_missing_tree = MultiwayTreeExtractor(target='FPD', features=['商品类别'], max_depth=1, max_branches=5, n_jobs=1).fit(train)
category_missing_tree.manual_split(feature='商品类别', category_groups=ordinary_groups + [[np.nan]])
assert category_missing_tree.nodes_[0].missing_separate
fig = plot_tree(category_missing_tree, save=str(output_dir / '类别独立缺失分支.png'))
merged_groups = copy.deepcopy(ordinary_groups)
merged_groups[0].append(np.nan)
category_missing_tree.manual_split(feature='商品类别', category_groups=merged_groups)
assert not category_missing_tree.nodes_[0].missing_separate
display(category_missing_tree.get_node_table())

## 11. 训练时无缺失，生产中新出现缺失

未指定缺失策略时，按当前节点的训练样本判断：有缺失则学习合并方向，无缺失则默认走最左（0 号）分支。
显式 `missing_branch`、`thresholds` 中的缺失标记和 `missing_separate=True` 始终按声明生效。
下面在真实数据副本中模拟生产缺失，并核对叶节点、预测概率与导出规则。
已保存模型保留原有路由；更新库后需重启内核、重新训练或重新分裂节点。

In [ ]:
complete_training = train.dropna(subset=[features[0]]).copy()
future_tree = MultiwayTreeExtractor(
    target='FPD', features=features, max_depth=1, max_branches=4, n_jobs=1,
).fit(complete_training)
future_tree.manual_split(feature=features[0], thresholds=[0.12, 0.18, 0.25])
future_root = future_tree.nodes_[0]
assert future_root.missing_branch == 0
production = complete_training.iloc[:5].copy()
production[features[0]] = np.nan
assert np.all(future_tree.apply(production) == future_root.children[0])
left_training = complete_training.loc[complete_training[features[0]] <= 0.12]
np.testing.assert_allclose(future_tree.predict_proba(production)[:, 1], left_training['FPD'].mean())
for node_id, rule in zip(future_tree.get_leaf_node_ids(), future_tree.get_rules()):
    np.testing.assert_array_equal(rule.predict(production).to_numpy(), future_tree.apply(production) == node_id)
explicit_future = copy.deepcopy(future_tree)
explicit_future.manual_split(feature=features[0], thresholds=[0.12, 0.18, 0.25], missing_branch=2)
assert np.all(explicit_future.apply(production) == explicit_future.nodes_[0].children[2])
fig = plot_tree(future_tree, save=str(output_dir / '训练无缺失默认走最左分支.png'))
display(style_rule_table(future_tree.get_rule_table(production, leaf_only=True)))